In [35]:
from torch.utils.data import Dataset,DataLoader
from torch import nn
import os
from PIL import Image
import torch
from random import randint
from tqdm import tqdm
from torchvision import transforms

In [36]:
class Boot_Rotate_Dataset(Dataset):
  def __init__(self,path):
    super(Boot_Rotate_Dataset,self).__init__()

    dirs=[os.path.join(path,dir) for dir in os.listdir(path)]

    self.all_images=[]
    for dir in dirs:
      images=[os.path.join(dir,img) for img in os.listdir(dir) if 'done' in img]
      self.all_images+=images

    self.trans=transforms.Compose([

        transforms.Resize((762,1100))
    ])
    self.tensor_trans=transforms.Compose([

        transforms.Resize((762,1100)),
        transforms.ToTensor()
    ])


  def __len__(self):
    return len(self.all_images)

  def __getitem__(self,idx):

    img=Image.open(self.all_images[idx])

    if randint(0,1):
      degr=randint(0,10)
      img=img.rotate(degr,expand=True)
      #img=self.trans(img)
      new_width,new_height=(610,932)

    else:
      degr=randint(350,360)
      img=img.rotate(degr,expand=True)
      #img=self.trans(img)
      new_width,new_height=(610,932)

    width,height=img.size
    left=(width-new_width)//2
    top=(height-new_height)//2
    right=left+new_width
    bottom=top+new_height
    img=img.crop((left,top,right,bottom))
    tensor_img=self.tensor_trans(img)
    return {
        "img": tensor_img,
        "label":torch.FloatTensor([degr])
        }

In [37]:
def Train_degr_model(model,dataloader,loss_func,optimizer,device):
    #loss_item=0#костыль
    model=model.to(device)
    sigm=nn.Sigmoid()
    for batch in (pbar:=tqdm(dataloader)):
        optimizer.zero_grad()
        pred=sigm(model(batch['img'].to(device)))
        #print(pred,batch['label'])


        loss=loss_func(pred,batch['label'].to(device))
        loss_item=loss.item()
        loss.backward()
        optimizer.step()
        pbar.set_description(f'loss: {loss_item}')
        try:
            torch.save(model.state_dict(),f'/content/drive/My Drive/Colab Notebooks/degr_net_{device}.pth')
        except:
            print('ошибка сохранения весов')

In [38]:
class Degree_Net(nn.Module):
    def __init__(self,input_size,hidden_size):
        super(Degree_Net,self).__init__()
    
        self.lay0=nn.Sequential(
            nn.Conv2d(input_size,hidden_size,3,padding=1),
            nn.BatchNorm2d(hidden_size),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay1=nn.Sequential(
            nn.Conv2d(hidden_size,hidden_size*2,3,padding=1),
        
            nn.BatchNorm2d(hidden_size*2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay2=nn.Sequential(
            nn.Conv2d(hidden_size*2,hidden_size*4,3,padding=1),
            nn.BatchNorm2d(hidden_size*4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.lay3=nn.Sequential(
            nn.Conv2d(hidden_size*4,hidden_size*8,3,padding=1),
            nn.BatchNorm2d(hidden_size*8),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2,2)
        )
        
        self.linear_lay=nn.Sequential(
            nn.Flatten(),
            nn.Linear(3214080,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x):
        print(x.shape)
        x0=self.lay0(x)
        print(x0.shape)
    
        x1=self.lay1(x0)
          
        print(x1.shape)
    
        x2=self.lay2(x1)
        print(x2.shape)
    
        x3=self.lay3(x2)
        print(x3.shape)
        
    
        final_x=self.linear_lay(x2)
        print(final_x.shape)
    
        return final_x

In [39]:
degr_dataset=Boot_Rotate_Dataset('/home/artemybombastic/MyGit/KD_Data/TransformData')
degr_dataloader=DataLoader(degr_dataset,batch_size=4,shuffle=True,drop_last=True)

In [40]:
model=Degree_Net(3,64)
loss_func=nn.MSELoss()
optimizer=torch.optim.AdamW(model.parameters())
device='cpu'

In [41]:
Train_degr_model(model=model,dataloader=degr_dataloader,loss_func=loss_func,optimizer=optimizer,device=device)

  0%|                                                             | 0/323 [00:00<?, ?it/s]

torch.Size([4, 3, 762, 1100])
torch.Size([4, 64, 381, 550])
torch.Size([4, 128, 190, 275])
torch.Size([4, 256, 95, 137])


  0%|                                                             | 0/323 [00:03<?, ?it/s]

torch.Size([4, 512, 47, 68])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x3331840 and 3214080x128)